In [1]:
import torch
import torchaudio

In [2]:
# Ambil modelnya
bundle = torchaudio.pipelines.WAV2VEC2_ASR_BASE_960H
model = bundle.get_model()

In [3]:
# Wave, sample rate dari audio file
import soundfile as sf

# torchaudio.load needs the torchcodec package in recent versions, so read the file with soundfile
data, sample_rate = sf.read("speech.wav", dtype="float32", always_2d=True)
wave = torch.from_numpy(data.T)  # (channels, samples), same layout as torchaudio.load

In [4]:
# Pre-process wave dan sample ratenya
if sample_rate != bundle.sample_rate:
    wave = torchaudio.functional.resample(wave, sample_rate, bundle.sample_rate)

In [5]:
with torch.inference_mode():
    emission, _ = model(wave)
print(emission)

tensor([[[15.2334, -1.8698, -3.2884,  ..., -6.7966, -6.9653, -7.4779],
         [15.2039, -1.9884, -3.3543,  ..., -6.7460, -6.8552, -7.5177],
         [15.2478, -2.0606, -3.2647,  ..., -6.7416, -6.8245, -7.5299],
         ...,
         [14.9047, -2.5286, -4.2227,  ..., -5.6411, -6.3297, -7.5230],
         [15.1738, -1.5068, -3.0886,  ..., -7.0477, -7.5659, -7.5628],
         [15.1209, -0.2923, -2.9573,  ..., -7.3561, -7.9182, -7.5748]]])


In [6]:
# Ubah jadi teks
# _labels = ('|', 'E', 'T', 'A', 'O', 'N', 'I', 'H', 'S', 'R')
class GreedyCTCDecoder(torch.nn.Module):
    # Constructor
    def __init__(self, labels, blank=0):
        super().__init__()
        self.labels = labels
        self.blank = blank

    def forward(self, emission: torch.Tensor) -> str:
        indices = torch.argmax(emission, dim=-1)
        # print(indices)
        indices = torch.unique_consecutive(indices, dim=-1)
        # print(indices)
        indices = [i for i in indices if i != self.blank]
        # print(indices)
        return "".join(self.labels[i] for i in indices)

In [7]:
decoder = GreedyCTCDecoder(labels=bundle.get_labels())
sentence = decoder(emission[0])
print(sentence)

I|HAD|THAT|CURIOSITY|BESIDE|ME|AT|THIS|MOMENT|
